<a href="https://colab.research.google.com/github/badBrock/MultiHeadAttentionJourney/blob/main/Revision_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
import re

result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]

In [ ]:
result[:1]

['I']

In [ ]:
vocab = {i:j for j,i in enumerate(sorted(set(result)))}

In [ ]:
class tokenizer():
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {j:i for i,j in vocab.items()}

    if "<|unk|>" not in self.str_to_int:
      unk_id = len(self.str_to_int)
      self.str_to_int["<|unk|>"] = unk_id
      self.int_to_str[unk_id] = "<|unk|>"

  def encode(self,txt):
    preprocessing = re.split(r'([,.:;?_!"()\']|--|\s)', txt)
    preprocessing = [item.strip() for item in preprocessing if item.strip()]

    preprocessing = [item if item in self.str_to_int else "<|unk|>" for item in preprocessing]
    return [self.str_to_int[x] for x in preprocessing]

  def decode(self,x):
    x = ' '.join([self.int_to_str[i] for i in x])
    text = re.split(r'([,.:;?_!"()\']|--|\s)', x)
    return text

In [ ]:
tokenizerz = tokenizer(vocab)

tokenizerz.encode("Hello, do you like tea?")

[1130, 5, 355, 1126, 628, 975, 10, 1130]

In [ ]:
class GPTDataset():
  def __init__(self,txt,tokenizer,context_length,stride):
    self.input_block = []
    self.output_block = []

    token_id = tokenizer.encode(txt)
    for token in range(0,len(token_id) - context_length, stride):
      input = token_id[token:token + context_length]
      output = token_id[token+1:token + context_length + 1]
      self.input_block.append(torch.tensor(input))
      self.output_block.append(torch.tensor(output))
  def __len__(self):
    return len(self.input_block)
  def __getitem__(self,idx):
    return self.input_block[idx], self.output_block[idx]

In [ ]:
import tiktoken
from torch.utils.data import DataLoader
def dataloader(txt,batch_size,context_length,stride,shuffle=True,drop_last=True,num_workers=0):
  tokenizer = tiktoken.get_encoding('gpt2')
  dataset = GPTDataset(txt,tokenizer,context_length,stride)

  dataload = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)

  return dataload

In [ ]:
vocab = 50000
batch = 4
context_length = 4
output_dim = 256

embedd = torch.nn.Embedding(vocab,output_dim)
pos = torch.nn.Embedding(context_length,output_dim)

In [ ]:
raw_text[:30]

'I HAD always thought Jack Gisb'

In [ ]:
load = dataloader(raw_text,batch_size=batch,context_length=context_length,stride=context_length)

In [ ]:
for batch in load:
  x,y = batch

  embeddings = embedd(x)
  pos_embedding = pos(torch.arange(context_length))

  input_embeddings = embeddings + pos_embedding
  break

In [ ]:
print(input_embeddings.shape)

torch.Size([4, 4, 256])
